# Side-Channel Attack System

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy.fft import fft, fftfreq
from scipy.signal import welch, find_peaks
import hashlib
from collections import defaultdict

class SideChannelAttackDetector:
    def __init__(self):
        self.power_model = IsolationForest(contamination=0.1, random_state=42)
        self.timing_model = IsolationForest(contamination=0.08, random_state=42)
        self.scaler = StandardScaler()
        self.baseline_profiles = defaultdict(list)

    def analyze_power_traces(self, power_data, operations):
        features = []
        for i, trace in enumerate(power_data):
            power_spectrum = np.abs(fft(trace))
            peak_frequencies = self.extract_frequency_peaks(trace)
            hamming_weight = bin(operations[i]).count('1') if i < len(operations) else 0
            variance = np.var(trace)
            correlation = np.corrcoef(trace[:-1], trace[1:])[0,1] if len(trace) > 1 else 0
            features.append([np.mean(power_spectrum), len(peak_frequencies), hamming_weight, variance, correlation])
        return np.array(features)

    def extract_frequency_peaks(self, signal, threshold=0.7):
        freqs, psd = welch(signal, nperseg=min(256, len(signal)//4))
        peaks, _ = find_peaks(psd, height=threshold * np.max(psd))
        return freqs[peaks]

    def analyze_timing_attacks(self, execution_times, input_data):
        features = []
        for i, time_trace in enumerate(execution_times):
            if i < len(input_data):
                input_entropy = self.calculate_entropy(str(input_data[i]).encode())
                time_variance = np.var(time_trace)
                cache_misses = self.estimate_cache_behavior(time_trace)
                branch_predictions = self.analyze_branch_patterns(time_trace)
                features.append([np.mean(time_trace), time_variance, input_entropy, cache_misses, branch_predictions])
        return np.array(features) if features else np.array([[0, 0, 0, 0, 0]])

    def calculate_entropy(self, data):
        if not data: return 0
        byte_counts = np.bincount(np.frombuffer(data, dtype=np.uint8))
        probs = byte_counts / len(data)
        return -np.sum(probs * np.log2(probs + 1e-10))

    def estimate_cache_behavior(self, timing_data):
        cache_miss_threshold = np.percentile(timing_data, 90)
        return np.sum(timing_data > cache_miss_threshold) / len(timing_data)

    def analyze_branch_patterns(self, timing_data):
        diff_sequence = np.diff(timing_data)
        pattern_changes = np.sum(np.abs(np.diff(np.sign(diff_sequence))))
        return pattern_changes / max(len(diff_sequence) - 1, 1)

    def detect_differential_power_analysis(self, power_traces, plaintexts, key_hypothesis):
        correlations = []
        for key_byte in range(256):
            hypothetical_power = []
            for i, plaintext in enumerate(plaintexts):
                intermediate = plaintext ^ key_byte
                hamming_weight = bin(intermediate).count('1')
                hypothetical_power.append(hamming_weight)

            if len(power_traces) == len(hypothetical_power):
                correlation = np.corrcoef(power_traces, hypothetical_power)[0,1]
                correlations.append(abs(correlation) if not np.isnan(correlation) else 0)

        max_correlation = max(correlations) if correlations else 0
        suspected_key = correlations.index(max_correlation) if correlations else 0
        return max_correlation > 0.7, suspected_key, max_correlation

    def train_baseline(self, benign_power_data, benign_timing_data, benign_operations):
        power_features = self.analyze_power_traces(benign_power_data, benign_operations)
        timing_features = self.analyze_timing_attacks(benign_timing_data, benign_operations)

        if len(power_features) > 0:
            self.power_model.fit(self.scaler.fit_transform(power_features))
        if len(timing_features) > 0:
            self.timing_model.fit(self.scaler.fit_transform(timing_features))

    def detect_side_channel_attack(self, power_data, timing_data, operations, plaintexts):
        power_anomalies = self.power_model.predict(self.scaler.transform(self.analyze_power_traces(power_data, operations)))
        timing_anomalies = self.timing_model.predict(self.scaler.transform(self.analyze_timing_attacks(timing_data, operations)))
        dpa_detected, key, correlation = self.detect_differential_power_analysis(np.mean(power_data, axis=1), plaintexts, 0)

        attack_probability = (np.mean(power_anomalies == -1) + np.mean(timing_anomalies == -1)) / 2
        return attack_probability > 0.3 or dpa_detected, {'key': key, 'correlation': correlation, 'probability': attack_probability}